In [ ]:
# ============================================================
# Mule Account Detection using LightGBM
# ============================================================
# Objective:
# This program detects suspicious mule accounts from banking data.
# The target column is F3924:
#   0 = Normal account
#   1 = Suspicious / mule account
#
# Since the dataset is highly imbalanced, we do not rely on accuracy.
# Instead, we evaluate using ROC-AUC, PR-AUC, Recall, F1-score,
# and Confusion Matrix.
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. Load Data
# ============================================================

DATA_PATH = "DataSet.csv"
TARGET = "F3924"

# Read dataset
df = pd.read_csv(DATA_PATH)

# Display basic dataset information
print("Original shape:", df.shape)
print("Target distribution:")
print(df[TARGET].value_counts())

# ============================================================
# 2. Basic Cleaning
# ============================================================

drop_cols = []

# Remove index-like column if present
if "Unnamed: 0" in df.columns:
    drop_cols.append("Unnamed: 0")

# Remove suspected leakage / ID-like column
if "F2230" in df.columns:
    drop_cols.append("F2230")

# Remove columns that are completely empty
empty_cols = df.columns[df.isna().mean() == 1].tolist()
drop_cols += empty_cols

# Drop selected columns
df = df.drop(columns=list(set(drop_cols)), errors="ignore")

# Separate target and input features
y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

# ============================================================
# 3. Feature Engineering
# ============================================================

# Convert date column into numerical "days old" feature
# This makes date information usable for ML models.
if "F3888" in X.columns:
    date_col = pd.to_datetime(X["F3888"], errors="coerce")
    max_date = date_col.max()
    X["F3888_days_old"] = (max_date - date_col).dt.days
    X = X.drop(columns=["F3888"])

# Extract numerical duration and type from encoded values like G365D
# Example:
#   G365D -> F3889_days = 365, F3889_type = G
if "F3889" in X.columns:
    X["F3889_days"] = X["F3889"].astype(str).str.extract(r"(\d+)").astype(float)
    X["F3889_type"] = X["F3889"].astype(str).str[0]

# Calculate missing value percentage for each feature
missing_rate = X.isna().mean()

# Drop features with more than 95% missing values
# Such features usually add noise and very little useful information.
high_missing_cols = missing_rate[missing_rate > 0.95].index.tolist()
X = X.drop(columns=high_missing_cols)

# Recalculate missing rate after dropping highly missing columns
missing_rate = X.isna().mean()

# Select columns where missingness may itself be useful
missing_cols = missing_rate[
    (missing_rate > 0.05) & (missing_rate < 0.95)
].index.tolist()

# Create missing-value indicator columns
# Example:
#   F100_missing = 1 if F100 is missing else 0
# This helps the model learn patterns from incomplete information.
if len(missing_cols) > 0:
    missing_indicators = X[missing_cols].isna().astype("int8")
    missing_indicators.columns = [col + "_missing" for col in missing_cols]
    X = pd.concat([X, missing_indicators], axis=1)

# Defragment dataframe after adding many columns
X = X.copy()

# Remove constant columns because they do not help prediction
constant_cols = X.columns[X.nunique(dropna=False) <= 1].tolist()
X = X.drop(columns=constant_cols)

# Identify categorical columns
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

# Convert categorical columns into category type for LightGBM
for col in cat_cols:
    X[col] = X[col].astype("category")

print("Final shape:", X.shape)
print("Categorical columns:", cat_cols)

# ============================================================
# 4. Model Training Setup
# ============================================================

# Stratified K-Fold keeps the same ratio of normal and mule accounts
# in each fold. This is important because the dataset is imbalanced.
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Store out-of-fold predictions
# These predictions are made only on validation folds.
oof_pred = np.zeros(len(X))

# Store trained models from each fold
models = []

# Calculate imbalance weight
# Normal accounts are much more frequent than mule accounts.
scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

# LightGBM parameters
params = {
    "objective": "binary",
    "metric": ["auc", "binary_logloss"],
    "boosting_type": "gbdt",

    # Learning rate controls how fast the model learns
    "learning_rate": 0.02,

    # Tree complexity parameters
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 20,

    # Random feature and row sampling to reduce overfitting
    "feature_fraction": 0.75,
    "bagging_fraction": 0.75,
    "bagging_freq": 5,

    # Regularization
    "lambda_l1": 1.0,
    "lambda_l2": 5.0,

    # Handles class imbalance
    "scale_pos_weight": scale_pos_weight,

    "verbose": -1,
    "seed": 42
}

# ============================================================
# 5. Train LightGBM using 5-Fold Cross Validation
# ============================================================

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n========== Fold {fold} ==========")

    # Split data into training and validation fold
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # LightGBM dataset format
    train_data = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    val_data = lgb.Dataset(
        X_val,
        label=y_val,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    # Train model with early stopping
    # Early stopping prevents overfitting by stopping training when
    # validation score stops improving.
    model = lgb.train(
        params,
        train_data,
        valid_sets=[val_data],
        valid_names=["valid"],
        num_boost_round=3000,
        callbacks=[
            lgb.early_stopping(150),
            lgb.log_evaluation(100)
        ]
    )

    # Predict probability of mule account for validation fold
    pred = model.predict(X_val, num_iteration=model.best_iteration)

    # Store out-of-fold predictions
    oof_pred[val_idx] = pred

    # Save model
    models.append(model)

    # Fold-level performance
    print("Fold ROC-AUC:", roc_auc_score(y_val, pred))
    print("Fold PR-AUC:", average_precision_score(y_val, pred))

# ============================================================
# 6. Overall Model Evaluation
# ============================================================

print("\n========== Overall Metrics ==========")

print("Overall ROC-AUC:", roc_auc_score(y, oof_pred))
print("Overall PR-AUC:", average_precision_score(y, oof_pred))

# ============================================================
# 7. Best F1 Threshold Search
# ============================================================

# Default threshold 0.5 is usually not ideal for fraud detection.
# We test many thresholds and select the one with best F1-score.

best_f1 = 0
best_threshold = 0.5

for t in np.arange(0.01, 0.99, 0.01):
    pred_label = (oof_pred >= t).astype(int)
    score = f1_score(y, pred_label)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("\nBest F1 Threshold:", best_threshold)
print("Best F1:", best_f1)

best_pred = (oof_pred >= best_threshold).astype(int)

print("\nClassification Report at Best F1 Threshold:")
print(classification_report(y, best_pred))

print("\nConfusion Matrix at Best F1 Threshold:")
print(confusion_matrix(y, best_pred))

# ============================================================
# 8. Threshold Test to Catch All Mule Accounts
# ============================================================
# This section tests lower thresholds to see whether all 81 mule
# accounts can be detected.
#
# Lower threshold:
#   - increases recall
#   - detects more mule accounts
#   - also increases false positives

print("\n========== Threshold Test ==========")

thresholds = [
    0.10, 0.09, 0.08, 0.07, 0.06,
    0.05, 0.04, 0.03, 0.02, 0.01,
    0.005, 0.001
]

best_recall_threshold = None

for t in thresholds:
    preds = (oof_pred >= t).astype(int)

    cm = confusion_matrix(y, preds)
    tn, fp, fn, tp = cm.ravel()

    print("\n------------------------------")
    print("Threshold:", t)
    print("Confusion Matrix:")
    print(cm)
    print("Mule detected:", tp, "/ 81")
    print("Mule missed:", fn)
    print("False positives:", fp)
    print("Precision:", precision_score(y, preds, zero_division=0))
    print("Recall:", recall_score(y, preds, zero_division=0))
    print("F1:", f1_score(y, preds, zero_division=0))

    # Store first threshold that catches all mule accounts
    if fn == 0 and best_recall_threshold is None:
        best_recall_threshold = t

# Choose final threshold
# If we find a threshold that catches all mule accounts, use it.
# Otherwise, use the best F1 threshold.
if best_recall_threshold is not None:
    final_threshold = best_recall_threshold
    print("\nFinal threshold chosen to catch all mule accounts:", final_threshold)
else:
    final_threshold = best_threshold
    print("\nNo tested threshold caught all mule accounts.")
    print("Using best F1 threshold:", final_threshold)

# ============================================================
# 9. Final Prediction
# ============================================================

final_pred = (oof_pred >= final_threshold).astype(int)

print("\n========== Final Model Report ==========")
print("Final Threshold:", final_threshold)
print(classification_report(y, final_pred))
print("Confusion Matrix:")
print(confusion_matrix(y, final_pred))

# ============================================================
# 10. Risk Score Output
# ============================================================

# Convert probability to risk score from 0 to 100.
# Example:
#   probability = 0.91 -> risk_score = 91

risk_output = pd.DataFrame({
    "actual_label": y.values,
    "risk_probability": oof_pred,
    "risk_score_0_100": (oof_pred * 100).round(2),
    "predicted_label": final_pred
})

risk_output.to_csv("final_mule_account_risk_scores.csv", index=False)

print("\nSaved: final_mule_account_risk_scores.csv")

# ============================================================
# 11. Feature Importance
# ============================================================

# Average feature importance across all folds
feature_importance_values = np.mean(
    [model.feature_importance(importance_type="gain") for model in models],
    axis=0
)

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": feature_importance_values
}).sort_values(by="importance", ascending=False)

importance.to_csv("final_feature_importance.csv", index=False)

print("Saved: final_feature_importance.csv")

print("\nTop 20 important features:")
print(importance.head(20))